# Try the Jev-class models on a Colab GPU
One model on the GPU at a time. Load it, call it with text, an image and a video, see the response times, switch to another.
Everything runs and is called **from this notebook**. Run the cells top to bottom.

**Terms.** Colab's FAQ disallows "web service offerings not related to interactive compute", so this notebook does not open a public URL or keep the session alive. See the README.

## 1. Get the code

In [ ]:
# clones just this folder from GitHub (a sparse clone: the repo holds more than the notebook needs)
REPO = "https://github.com/mvp-scale/workspace"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
SUBDIR = "colab"  #@param {type:"string"}  # leave empty if the repo root is this folder
import os, subprocess
def sh(c): subprocess.run(c, shell=True, check=True)
sh("rm -rf /content/src")
sh(f"git clone -q --depth 1 --branch {BRANCH} " + (f"--filter=blob:none --sparse {REPO} /content/src && git -C /content/src sparse-checkout set {SUBDIR}" if SUBDIR else f"{REPO} /content/src"))
os.chdir(f"/content/src/{SUBDIR}")
print(os.getcwd(), sorted(os.listdir()))

## 2. Which GPU, and which models fit it

In [ ]:
import json, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > GPU")
import gateway
gpu, mib = gateway.gpu_info()
g = gateway.Gateway.__new__(gateway.Gateway); g.gpu, g.gpu_mib = gpu, mib
for m in json.load(open("models.json"))["models"]:
    if m.get("kind") in ("llama", "proc"):
        print(("fits   " if g.fits(m) else "NO FIT ") + f"{m['id']:<17} {m['vram_gib']:>5} GiB  {m['name']}")

## 3. Install the llama.cpp server (Clef Flash Q4 and Q2 use it)
A T4 has no ready-made code in the official build, so the first start compiles it (slow once). For speed, host the prebuilt `llama-b11430-t4.tar.gz` and put its URL below; see the README.

In [ ]:
LLAMA_T4_URL = ""  #@param {type:"string"}
import os
os.environ["LLAMA_T4_URL"] = LLAMA_T4_URL
LLAMA = !LLAMA_DIR=$PWD/llama ./setup_llama.sh
LLAMA_BIN = LLAMA[-1]
print(LLAMA_BIN)

## 4. Start the gateway and load a model
The first load of a model installs its packages and downloads its weights (minutes). Later loads are fast.

In [ ]:
MODEL = "clef-flash-q4km"  #@param ["clef-flash-q4km", "clef-flash-q2k", "clef-flash", "laya", "verdict", "semif", "so1", "kev-0.5b", "kev-0.8b", "kev-4b", "jeff"]
import subprocess, sys, time, urllib.request, json, secrets
KEY = secrets.token_urlsafe(12)
URL = "http://127.0.0.1:8000"
gw = subprocess.Popen([sys.executable, "gateway.py", "--port", "8000", "--key", KEY, "--llama-bin", LLAMA_BIN, "--root", os.path.abspath("vendor"), "--work", os.path.abspath("work")])
time.sleep(2)
import try_it
t = time.time(); status = try_it.select(URL, KEY, MODEL)
print("loaded:", status.get("loaded"), "in", round(time.time() - t), "s", status.get("error") or "")

## 5. Test it: text, image, video
Five timed calls each after one warm-up. Video goes in as sampled frames. Set `IMAGE` / `VIDEO` to your own files (use the Files panel to upload).

In [ ]:
IMAGE = ""  #@param {type:"string"}
VIDEO = ""  #@param {type:"string"}
import json
rows = try_it.demo(URL, KEY, IMAGE or None, VIDEO or None, frames=6, n=5)
for r in rows:
    print(r["test"], "->", json.dumps(r["answers"])[:200])

## 6. Switch to another model
The current model is unloaded first. A model that does not fit this GPU is refused with the reason.

In [ ]:
MODEL = "laya"  #@param ["clef-flash-q4km", "clef-flash-q2k", "clef-flash", "laya", "verdict", "semif", "so1", "kev-0.5b", "kev-0.8b", "kev-4b", "jeff"]
t = time.time(); status = try_it.select(URL, KEY, MODEL)
print("loaded:", status.get("loaded"), "in", round(time.time() - t), "s", status.get("error") or "")
rows = try_it.demo(URL, KEY, IMAGE or None, VIDEO or None, frames=6, n=5)

## 7. Stop
Frees the GPU.

In [ ]:
gw.terminate()